# Microproyecto 2 - Clasificacion de textos ODS (Experimentacion exhaustiva)

**App desplegada:** [https://ods-prediction-uniandes.streamlit.app/](https://ods-prediction-uniandes.streamlit.app/)

Este notebook implementa todo el plan de experimentacion:

- Preprocesamiento textual configurable: tokenizacion, normalizacion, stopwords, lematizacion y stemming.
- Representaciones: TF-IDF y embeddings (clasicos + Hugging Face).
- Reduccion de dimensionalidad: SVD, NMF y PCA.
- Modelos: Naive Bayes, CatBoost, LightGBM.
- Protocolo: split `train/val/test`, tuning sobre `train`, seleccion por `val`, evaluacion y ranking final en `test`.

El notebook selecciona el ganador y exporta artefactos en `artifacts/`.

## Justificacion de la metrica principal

Se utiliza **`f1_macro`** como metrica de seleccion porque, al tener 17 clases ODS con distribucion desigual, el accuracy seria engañoso: un modelo que favorezca las clases mayoritarias obtendria un accuracy alto ignorando las clases minoritarias. `f1_macro` calcula el F1 por clase y promedia sin ponderar, penalizando asi modelos que no clasifiquen bien **todas** las clases por igual.

## Justificacion del protocolo de evaluacion

Se emplea un split estratificado **train / val / test (70/15/15)** en lugar de validacion cruzada (CV). Con ~9600 textos y modelos costosos (CatBoost, LightGBM, embeddings), la CV multiplicaria el tiempo de ejecucion por el numero de folds. El holdout estratificado garantiza que cada particion preserve la proporcion de clases y permite una comparacion justa entre experimentos sin comprometer la viabilidad temporal.

In [1]:
import os
import re
import json
import time
import random
import warnings
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print('Seed:', SEED)

Seed: 42


In [ ]:
# Configuracion del experimento (parametros para ejecucion rapida)
USE_DEV_SAMPLE = False
DEV_SAMPLE_SIZE = int(os.getenv('DEV_SAMPLE_SIZE', '2400'))
FAST_MODE = True                      # Reduce candidatos: rand_iter=3 en lugar de 8
MAX_EXPERIMENTS = int(os.getenv('MAX_EXPERIMENTS', '0'))  # 0 => todos

ENABLE_HF_EXPERIMENTS = False         # Omite experimentos HF (E13, E14) que son lentos en CPU
HF_MODEL_NAME = os.getenv('HF_MODEL_NAME', 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
HF_N_ITER = int(os.getenv('HF_N_ITER', '2'))

LIMIT_PREPROC_VARIANTS = True         # Solo 2 variantes de preprocesamiento en lugar de 4

print('USE_DEV_SAMPLE:', USE_DEV_SAMPLE)
print('DEV_SAMPLE_SIZE:', DEV_SAMPLE_SIZE)
print('FAST_MODE:', FAST_MODE)
print('MAX_EXPERIMENTS:', MAX_EXPERIMENTS)
print('ENABLE_HF_EXPERIMENTS:', ENABLE_HF_EXPERIMENTS)
print('HF_MODEL_NAME:', HF_MODEL_NAME)
print('HF_N_ITER:', HF_N_ITER)
print('LIMIT_PREPROC_VARIANTS:', LIMIT_PREPROC_VARIANTS)

USE_DEV_SAMPLE: False
DEV_SAMPLE_SIZE: 2400
FAST_MODE: True
MAX_EXPERIMENTS: 0
ENABLE_HF_EXPERIMENTS: False
HF_MODEL_NAME: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
HF_N_ITER: 2
LIMIT_PREPROC_VARIANTS: True


In [3]:
# Import de librerias ML/NLP
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.model_selection import train_test_split, ParameterGrid, ParameterSampler
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD, NMF, PCA
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from sklearn.naive_bayes import ComplementNB, GaussianNB

from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

import nltk
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
import spacy

from gensim.models import FastText, Word2Vec
from sentence_transformers import SentenceTransformer

import torch
from IPython.display import display

print('Imports cargados correctamente.')

Imports cargados correctamente.


In [4]:
# Deteccion automatica de dispositivo (compatible con Mac/Windows/cloud)
import platform

cuda_available = torch.cuda.is_available()
mps_available = hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()

if cuda_available:
    HF_DEVICE = 'cuda'
    HF_BATCH_SIZE = 64
    _device_label = f'CUDA ({torch.cuda.get_device_name(0)})'
elif mps_available:
    HF_DEVICE = 'mps'
    HF_BATCH_SIZE = 32
    _device_label = 'MPS (Apple Silicon)'
else:
    HF_DEVICE = 'cpu'

    HF_BATCH_SIZE = 16
    print(f'HF_BATCH_SIZE: {HF_BATCH_SIZE}')

    _device_label = 'CPU'
    print(f'HF_DEVICE   : {HF_DEVICE}')

    print(f'Dispositivo : {_device_label}')

    print(f'Entorno     : {platform.system()} {platform.machine()}')
    print(f'torch       : {torch.__version__}')

HF_BATCH_SIZE: 16
HF_DEVICE   : cpu
Dispositivo : CPU
Entorno     : Windows AMD64
torch       : 2.10.0+cpu


In [5]:
# Recursos linguisticos
NLTK_DATA_DIR = Path('nltk_data')
NLTK_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Fuerza a NLTK a usar directorio local del proyecto (evita errores de permisos en HOME)
if str(NLTK_DATA_DIR.resolve()) not in nltk.data.path:
    nltk.data.path.insert(0, str(NLTK_DATA_DIR.resolve()))

nltk.download('stopwords', download_dir=str(NLTK_DATA_DIR.resolve()), quiet=True)
SPANISH_STOPWORDS = set(stopwords.words('spanish'))

_SPACY_MODEL = None

def get_spacy_model():
    global _SPACY_MODEL
    if _SPACY_MODEL is None:
        try:
            _SPACY_MODEL = spacy.load('es_core_news_sm', disable=['parser', 'ner'])
        except OSError:
            from spacy.cli import download
            download('es_core_news_sm')
            _SPACY_MODEL = spacy.load('es_core_news_sm', disable=['parser', 'ner'])
    return _SPACY_MODEL

print('Stopwords ES:', len(SPANISH_STOPWORDS))
print('NLTK_DATA_DIR:', NLTK_DATA_DIR.resolve())

Stopwords ES: 313
NLTK_DATA_DIR: C:\Users\wapon\Projects\GitHub\ods-prediction\nltk_data


## Carga y exploracion del dataset

El dataset OSDG-CD contiene textos en español asociados a los 17 ODS. Se eliminan registros nulos y se verifica la distribucion de clases. Conocer la distribucion es importante para confirmar que el desbalance justifica el uso de `f1_macro` como metrica y la eleccion de `ComplementNB` (diseñado para clases desbalanceadas) sobre `MultinomialNB`.

In [6]:
# Carga de datos
DATA_PATH = Path('data/train.xlsx')
if not DATA_PATH.exists():
    raise FileNotFoundError(f'No existe {DATA_PATH}')

df = pd.read_excel(DATA_PATH)
df = df.dropna(subset=['textos', 'ODS']).copy()
df['textos'] = df['textos'].astype(str)
df['ODS'] = df['ODS'].astype(int)

if USE_DEV_SAMPLE and DEV_SAMPLE_SIZE < len(df):
    df, _ = train_test_split(
        df,
        train_size=DEV_SAMPLE_SIZE,
        stratify=df['ODS'],
        random_state=SEED,
    )
    df = df.reset_index(drop=True)

print('Shape final:', df.shape)
print('ODS unicos:', sorted(df['ODS'].unique().tolist()))
print('Distribucion clases:')
print(df['ODS'].value_counts().sort_index())

df.head(3)

Shape final: (9656, 2)
ODS unicos: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
Distribucion clases:
ODS
1      505
2      369
3      894
4     1025
5     1070
6      695
7      787
8      446
9      343
10     352
11     607
12     312
13     464
14     377
15     330
16    1080
Name: count, dtype: int64


,textos,ODS
0,"""Aprendizaje"" y ""educación"" se consideran sinó...",4
1,No dejar clara la naturaleza de estos riesgos ...,6
2,"Como resultado, un mayor y mejorado acceso al ...",13


In [7]:
# Split train / val / test
X = df['textos']
y = df['ODS']

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y,
    test_size=0.15,
    stratify=y,
    random_state=SEED,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=0.17647,  # para 70/15/15 aprox
    stratify=y_train_val,
    random_state=SEED,
)

print('train:', len(X_train))
print('val  :', len(X_val))
print('test :', len(X_test))

train: 6758
val  : 1449
test : 1449


## Preprocesamiento textual

El pipeline de preprocesamiento textual (`TextPreprocessor`) encapsula en un solo transformador compatible con scikit-learn las operaciones de limpieza y normalizacion del texto. Se estudiaron estas tecnicas como buenas practicas para preparar texto antes de la vectorizacion:

- **Tokenizacion**: segmentacion del texto en tokens individuales mediante expresion regular, eliminando puntuacion y caracteres especiales.
- **Normalizacion**: conversion a minusculas y eliminacion de acentos (`NFKD`), reduciendo la variabilidad superficial.
- **Eliminacion de stopwords**: remocion de palabras funcionales (preposiciones, articulos) que no aportan contenido semantico relevante.
- **Lematizacion** (via spaCy): reduccion de cada palabra a su lema, preservando la semantica. Util cuando la forma canonica mejora la agrupacion de variantes morfologicas de una misma palabra.
- **Stemming** (via SnowballStemmer): reduccion agresiva a la raiz. Mas rapido que la lematizacion pero pierde informacion; se incluye para comparar empiricamente.

Estas cuatro variantes linguisticas (basic, stopwords, lemma+stop, stem+stop) se integran directamente en la busqueda de hiperparametros, de modo que el modelo seleccione automaticamente la combinacion de preprocesamiento mas efectiva para cada representacion y clasificador.

In [8]:
# Transformadores personalizados (autocontenidos en notebook)
import hashlib


class TextPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        lowercase=True,
        strip_accents=True,
        remove_stopwords=False,
        lemmatize=False,
        stem=False,
        min_token_len=2,
        stopword_language='spanish',
        spacy_model_name='es_core_news_sm',
        nltk_data_dir='nltk_data',
    ):
        self.lowercase = lowercase
        self.strip_accents = strip_accents
        self.remove_stopwords = remove_stopwords
        self.lemmatize = lemmatize
        self.stem = stem
        self.min_token_len = min_token_len
        self.stopword_language = stopword_language
        self.spacy_model_name = spacy_model_name
        self.nltk_data_dir = nltk_data_dir

    def fit(self, X, y=None):
        data_dir = Path(self.nltk_data_dir).resolve()
        data_dir.mkdir(parents=True, exist_ok=True)
        if str(data_dir) not in nltk.data.path:
            nltk.data.path.insert(0, str(data_dir))

        nltk.download('stopwords', download_dir=str(data_dir), quiet=True)
        self._stopwords = set(stopwords.words(self.stopword_language))
        self._stemmer = SnowballStemmer(self.stopword_language)

        if self.lemmatize:
            try:
                self._nlp = spacy.load(self.spacy_model_name, disable=['parser', 'ner'])
            except OSError:
                from spacy.cli import download
                download(self.spacy_model_name)
                self._nlp = spacy.load(self.spacy_model_name, disable=['parser', 'ner'])
        else:
            self._nlp = None

        return self

    def _normalize(self, text):
        text = str(text)
        if self.lowercase:
            text = text.lower()
        if self.strip_accents:
            text = ''.join(
                c for c in unicodedata.normalize('NFKD', text)
                if not unicodedata.combining(c)
            )
        return text

    def _tokenize(self, text):
        return re.findall(r'[a-zA-ZñÑüÜ0-9]+', text)

    def transform(self, X):
        processed = []
        for text in X:
            text = self._normalize(text)
            toks = self._tokenize(text)
            toks = [t for t in toks if len(t) >= self.min_token_len]

            if self.remove_stopwords:
                toks = [t for t in toks if t not in self._stopwords]

            if self.lemmatize and self._nlp is not None:
                doc = self._nlp(' '.join(toks))
                toks = [tok.lemma_.strip() for tok in doc if tok.lemma_.strip()]

            if self.stem:
                toks = [self._stemmer.stem(t) for t in toks]

            processed.append(' '.join(toks))

        return processed


class MeanGensimEmbeddingTransformer(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        method='word2vec',
        vector_size=150,
        window=5,
        min_count=2,
        epochs=20,
        workers=1,
        seed=42,
    ):
        self.method = method
        self.vector_size = vector_size
        self.window = window
        self.min_count = min_count
        self.epochs = epochs
        self.workers = workers
        self.seed = seed

    def fit(self, X, y=None):
        tokenized = [str(x).split() for x in X]
        if self.method == 'fasttext':
            self.model_ = FastText(
                sentences=tokenized,
                vector_size=self.vector_size,
                window=self.window,
                min_count=self.min_count,
                epochs=self.epochs,
                workers=self.workers,
                seed=self.seed,
            )
        else:
            self.model_ = Word2Vec(
                sentences=tokenized,
                vector_size=self.vector_size,
                window=self.window,
                min_count=self.min_count,
                epochs=self.epochs,
                workers=self.workers,
                seed=self.seed,
            )
        return self

    def transform(self, X):
        vectors = np.zeros((len(X), self.vector_size), dtype=np.float32)
        for i, text in enumerate(X):
            toks = str(text).split()
            vecs = [self.model_.wv[t] for t in toks if t in self.model_.wv]
            if vecs:
                vectors[i] = np.mean(vecs, axis=0)
        return vectors


class HFEmbeddingTransformer(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        model_name='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
        device='cpu',
        batch_size=32,
        normalize_embeddings=True,
        use_cache=True,
        cache_dir='cache/hf_embeddings',
    ):
        self.model_name = model_name
        self.device = device
        self.batch_size = batch_size
        self.normalize_embeddings = normalize_embeddings
        self.use_cache = use_cache
        self.cache_dir = cache_dir

    def fit(self, X, y=None):
        self.model_ = SentenceTransformer(self.model_name, device=self.device)
        return self

    def _build_cache_path(self, X):
        hasher = hashlib.sha1()
        hasher.update(self.model_name.encode('utf-8'))
        hasher.update(str(self.normalize_embeddings).encode('utf-8'))
        hasher.update(str(len(X)).encode('utf-8'))
        for text in X:
            s = str(text)
            hasher.update(s.encode('utf-8', errors='ignore'))
            hasher.update(b'\x00')
        return Path(self.cache_dir) / f"{hasher.hexdigest()}.npy"

    def transform(self, X):
        texts = list(map(str, X))

        if self.use_cache:
            cache_path = self._build_cache_path(texts)
            if cache_path.exists():
                return np.load(cache_path).astype(np.float32)

        if not hasattr(self, 'model_'):
            self.model_ = SentenceTransformer(self.model_name, device=self.device)

        emb = self.model_.encode(
            texts,
            batch_size=self.batch_size,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=self.normalize_embeddings,
        )
        emb = emb.astype(np.float32)

        if self.use_cache:
            cache_path.parent.mkdir(parents=True, exist_ok=True)
            np.save(cache_path, emb)

        return emb




In [9]:
# Preview de preprocessing
preview = pd.DataFrame({'raw': X_train.reset_index(drop=True).head(3)})

preview['basic'] = TextPreprocessor(remove_stopwords=False, lemmatize=False, stem=False).fit_transform(preview['raw'])
preview['stop'] = TextPreprocessor(remove_stopwords=True, lemmatize=False, stem=False).fit_transform(preview['raw'])
preview['lemma_stop'] = TextPreprocessor(remove_stopwords=True, lemmatize=True, stem=False).fit_transform(preview['raw'])
preview['stem_stop'] = TextPreprocessor(remove_stopwords=True, lemmatize=False, stem=True).fit_transform(preview['raw'])

preview

,raw,basic,stop,lemma_stop,stem_stop
0,"Tirpak et al (2010) recomiendan que, dada la n...",tirpak et al 2010 recomiendan que dada la natu...,tirpak et 2010 recomiendan dada naturaleza con...,tirpak et 2010 recomendar dado naturaleza cont...,tirpak et 2010 recomiend dad naturalez context...
1,Un liderazgo equilibrado y una formulación de ...,un liderazgo equilibrado una formulacion de po...,liderazgo equilibrado formulacion politicas se...,liderazgo equilibrado formulacion politica sen...,liderazg equilibr formul polit sensibl gener m...
2,Si bien la generalización de las fallas graves...,si bien la generalizacion de las fallas graves...,si bien generalizacion fallas graves mercado s...,si bien generalizacion falla grave mercado sec...,si bien generaliz fall grav merc sector atenci...


## Estrategia de representacion y reduccion de dimensionalidad

### Representacion: TF-IDF como esquema BOW

`TfidfVectorizer` de scikit-learn implementa en un solo paso la construccion de la Bolsa de Palabras (BOW) mediante conteo de frecuencias (`CountVectorizer`) y la ponderacion TF-IDF (`TfidfTransformer`). Es decir, `TfidfVectorizer` es equivalente a la secuencia `CountVectorizer → TfidfTransformer`, pero mas eficiente en memoria y computo. Se ajustan hiperparametros como `ngram_range`, `min_df` y `sublinear_tf` para controlar el vocabulario y la escala de pesos.

### Reduccion de dimensionalidad en rama TF-IDF

- **TruncatedSVD**: se utiliza en lugar de PCA porque las matrices TF-IDF son dispersas (sparse). PCA requiere centrar los datos, lo que convertiria la matriz dispersa en densa, multiplicando el uso de memoria. TruncatedSVD opera directamente sobre matrices dispersas sin centrar, preservando la eficiencia.
- **NMF** (Non-negative Matrix Factorization): alternativa a SVD que opera nativamente sobre matrices no negativas (como TF-IDF). Produce componentes interpretables como "topicos" del corpus, ya que los coeficientes son siempre no negativos.

### Representacion: Embeddings

Como alternativa a BOW, se exploran representaciones densas:

- **Word2Vec** y **FastText** (via Gensim): entrenan embeddings de palabras sobre el corpus y representan cada documento como el promedio de los vectores de sus tokens. FastText, a diferencia de Word2Vec, descompone las palabras en n-gramas de caracteres, lo que le permite generar vectores para palabras fuera de vocabulario (OOV).
- **Sentence-Transformers** (Hugging Face): modelo preentrenado multilingue que genera un unico vector denso por documento, capturando relaciones semanticas complejas sin necesidad de promediar por tokens.

### Reduccion de dimensionalidad en embeddings

- **PCA**: a diferencia de la rama TF-IDF, los embeddings ya son vectores densos de baja-media dimensionalidad (100-384 dims), por lo que PCA es adecuado. Se utiliza para eliminar componentes de baja varianza (ruido) y, si aplica, reducir la dimensionalidad a un espacio mas manejable. Se varia `n_components` en la busqueda de hiperparametros para determinar la dimension optima.

In [10]:
# Utilidades de experimentacion (sin CV)

# Variantes linguisticas obligatorias
PREPROC_VARIANTS = [
    {'prep__remove_stopwords': [False], 'prep__lemmatize': [False], 'prep__stem': [False]},
    {'prep__remove_stopwords': [True],  'prep__lemmatize': [False], 'prep__stem': [False]},
    {'prep__remove_stopwords': [True],  'prep__lemmatize': [True],  'prep__stem': [False]},
    {'prep__remove_stopwords': [True],  'prep__lemmatize': [False], 'prep__stem': [True]},
]

if LIMIT_PREPROC_VARIANTS:
    PREPROC_VARIANTS = PREPROC_VARIANTS[:2]


def add_preproc_variants(base):
    spaces = []
    for variant in PREPROC_VARIANTS:
        p = dict(base)
        p.update(variant)
        spaces.append(p)
    return spaces


def build_param_candidates(param_space, search_kind, n_iter, seed):
    spaces = param_space if isinstance(param_space, list) else [param_space]

    if search_kind == 'grid':
        candidates = []
        for sp in spaces:
            candidates.extend(list(ParameterGrid(sp)))
    else:
        candidates = []
        n_spaces = len(spaces)
        base = max(1, n_iter // n_spaces)
        rem = max(0, n_iter - base * n_spaces)
        for i, sp in enumerate(spaces):
            n_local = base + (1 if i < rem else 0)
            n_local = max(1, n_local)
            sampled = list(ParameterSampler(sp, n_iter=n_local, random_state=seed + i))
            candidates.extend(sampled)

    # deduplicacion
    unique = []
    seen = set()
    for p in candidates:
        key = tuple(sorted((k, repr(v)) for k, v in p.items()))
        if key not in seen:
            seen.add(key)
            unique.append(p)

    return unique


def run_experiment(exp_cfg):
    name = exp_cfg['name']
    search_kind = exp_cfg.get('search_kind', 'random')
    n_iter = exp_cfg.get('n_iter', 6 if not FAST_MODE else 3)

    print(f"\n===== {name} =====")
    t0 = time.perf_counter()

    candidates = build_param_candidates(exp_cfg['params'], search_kind, n_iter, SEED)
    if len(candidates) == 0:
        raise RuntimeError(f'No se generaron candidatos para {name}')

    best_val = -1.0
    best_model = None
    best_params = None

    for params in candidates:
        model = clone(exp_cfg['pipeline'])
        model.set_params(**params)
        model.fit(X_train, y_train)

        y_val_pred = model.predict(X_val)
        val_f1 = float(f1_score(y_val, y_val_pred, average='macro'))

        if val_f1 > best_val:
            best_val = val_f1
            best_model = model
            best_params = params

    y_val_pred = best_model.predict(X_val)
    y_test_pred = best_model.predict(X_test)

    elapsed = time.perf_counter() - t0

    row = {
        'experiment': name,
        'status': 'ok',
        'val_f1_macro': float(f1_score(y_val, y_val_pred, average='macro')),
        'val_f1_weighted': float(f1_score(y_val, y_val_pred, average='weighted')),
        'val_accuracy': float(accuracy_score(y_val, y_val_pred)),
        'test_f1_macro': float(f1_score(y_test, y_test_pred, average='macro')),
        'test_f1_weighted': float(f1_score(y_test, y_test_pred, average='weighted')),
        'test_accuracy': float(accuracy_score(y_test, y_test_pred)),
        'fit_time_sec': round(elapsed, 2),
        'n_candidates': len(candidates),
        'best_params': best_params,
    }

    print('VAL f1_macro :', round(row['val_f1_macro'], 4))
    print('TEST f1_macro:', round(row['test_f1_macro'], 4))
    print('Candidatos   :', row['n_candidates'])
    print('Tiempo (s)   :', row['fit_time_sec'])

    result = {
        'best_model': best_model,
        'best_params': best_params,
        'n_candidates': len(candidates),
    }

    return row, result

## Justificacion de los modelos seleccionados

Se comparan tres familias de clasificadores:

1. **Naive Bayes**: modelo probabilistico eficiente, sirve como baseline solido para clasificacion de textos.
   - `ComplementNB` para TF-IDF: diseñado para datasets con clases desbalanceadas; estima parametros usando las clases **complementarias**, reduciendo el sesgo hacia clases mayoritarias.
   - `GaussianNB` para embeddings densos: asume distribucion normal de las features, adecuado para vectores continuos (Word2Vec, FastText, HF).

2. **CatBoost**: gradient boosting con manejo nativo de features categoricas y regularizacion interna. Se usa con `loss_function='MultiClass'` para clasificacion multiclase directa.

3. **LightGBM**: gradient boosting basado en histogramas, eficiente en memoria y velocidad. Con `objective='multiclass'` y parametros como `num_leaves` y `learning_rate` en la busqueda.

La busqueda de hiperparametros se implementa con `ParameterSampler` (random search), evaluando cada combinacion de preprocesamiento + representacion + reduccion + clasificador sobre el conjunto de validacion para seleccionar la mejor configuracion por experimento.

In [11]:
# Matriz de experimentos
num_classes = y_train.nunique()
cb_iters = [200, 350] if not FAST_MODE else [120, 200]
lgb_estimators = [250, 450] if not FAST_MODE else [150, 250]
rand_iter = 8 if not FAST_MODE else 3

experiments = [
    # --- Rama TF-IDF ---
    {
        'name': 'E01_Preproc_TFIDF_ComplementNB',
        'search_kind': 'random',
        'n_iter': rand_iter,
        'pipeline': Pipeline([
            ('prep', TextPreprocessor()),
            ('tfidf', TfidfVectorizer(max_features=50000)),
            ('clf', ComplementNB()),
        ]),
        'params': add_preproc_variants({
            'tfidf__ngram_range': [(1, 1), (1, 2)],
            'tfidf__min_df': [2, 5],
            'tfidf__sublinear_tf': [True, False],
            'clf__alpha': [0.3, 0.7, 1.0],
        }),
    },
    {
        'name': 'E01b_Preproc_TFIDF_SVD_ComplementNB',
        'search_kind': 'random',
        'n_iter': rand_iter,
        'pipeline': Pipeline([
            ('prep', TextPreprocessor()),
            ('tfidf', TfidfVectorizer(max_features=50000)),
            ('svd', TruncatedSVD(random_state=SEED)),
            ('clf', ComplementNB()),
        ]),
        'params': add_preproc_variants({
            'tfidf__ngram_range': [(1, 1), (1, 2)],
            'tfidf__min_df': [2, 5],
            'tfidf__sublinear_tf': [True],
            'svd__n_components': [120, 220, 320],
            'clf__alpha': [0.3, 0.7, 1.0],
        }),
    },
    {
        'name': 'E02_Preproc_TFIDF_SVD_CatBoost',
        'search_kind': 'random',
        'n_iter': rand_iter,
        'pipeline': Pipeline([
            ('prep', TextPreprocessor()),
            ('tfidf', TfidfVectorizer(max_features=50000)),
            ('svd', TruncatedSVD(random_state=SEED)),
            ('clf', CatBoostClassifier(
                loss_function='MultiClass',
                random_state=SEED,
                verbose=0,
                allow_writing_files=False,
                thread_count=-1,
            )),
        ]),
        'params': add_preproc_variants({
            'tfidf__ngram_range': [(1, 1), (1, 2)],
            'tfidf__min_df': [2, 5],
            'svd__n_components': [120, 220, 320],
            'clf__depth': [6, 8],
            'clf__learning_rate': [0.03, 0.05, 0.1],
            'clf__iterations': cb_iters,
        }),
    },
    {
        'name': 'E03_Preproc_TFIDF_SVD_LightGBM',
        'search_kind': 'random',
        'n_iter': rand_iter,
        'pipeline': Pipeline([
            ('prep', TextPreprocessor()),
            ('tfidf', TfidfVectorizer(max_features=50000)),
            ('svd', TruncatedSVD(random_state=SEED)),
            ('clf', LGBMClassifier(
                objective='multiclass',
                num_class=num_classes,
                random_state=SEED,
                n_jobs=-1,
                verbosity=-1,
            )),
        ]),
        'params': add_preproc_variants({
            'tfidf__ngram_range': [(1, 1), (1, 2)],
            'tfidf__min_df': [2, 5],
            'svd__n_components': [120, 220, 320],
            'clf__num_leaves': [31, 63],
            'clf__learning_rate': [0.03, 0.05, 0.1],
            'clf__n_estimators': lgb_estimators,
        }),
    },
    {
        'name': 'E04_Preproc_TFIDF_NMF_LightGBM',
        'search_kind': 'random',
        'n_iter': rand_iter,
        'pipeline': Pipeline([
            ('prep', TextPreprocessor()),
            ('tfidf', TfidfVectorizer(max_features=50000)),
            ('nmf', NMF(random_state=SEED, init='nndsvda', max_iter=500)),
            ('clf', LGBMClassifier(
                objective='multiclass',
                num_class=num_classes,
                random_state=SEED,
                n_jobs=-1,
                verbosity=-1,
            )),
        ]),
        'params': add_preproc_variants({
            'tfidf__ngram_range': [(1, 1), (1, 2)],
            'tfidf__min_df': [2, 5],
            'nmf__n_components': [80, 140, 200],
            'clf__num_leaves': [31, 63],
            'clf__learning_rate': [0.03, 0.05, 0.1],
            'clf__n_estimators': lgb_estimators,
        }),
    },

    # --- Embeddings clasicos + Naive Bayes ---
    {
        'name': 'E05_Preproc_Word2Vec_GaussianNB',
        'search_kind': 'random',
        'n_iter': rand_iter,
        'pipeline': Pipeline([
            ('prep', TextPreprocessor()),
            ('emb', MeanGensimEmbeddingTransformer(method='word2vec', seed=SEED, workers=1)),
            ('clf', GaussianNB()),
        ]),
        'params': add_preproc_variants({
            'emb__vector_size': [100, 150, 200],
            'emb__window': [5, 8],
            'emb__min_count': [1, 2],
            'emb__epochs': [20, 30],
            'clf__var_smoothing': [1e-9, 1e-8, 1e-7],
        }),
    },
    {
        'name': 'E06_Preproc_Word2Vec_PCA_GaussianNB',
        'search_kind': 'random',
        'n_iter': rand_iter,
        'pipeline': Pipeline([
            ('prep', TextPreprocessor()),
            ('emb', MeanGensimEmbeddingTransformer(method='word2vec', vector_size=200, seed=SEED, workers=1)),
            ('pca', PCA(random_state=SEED)),
            ('clf', GaussianNB()),
        ]),
        'params': add_preproc_variants({
            'emb__window': [5, 8],
            'emb__min_count': [1, 2],
            'emb__epochs': [20, 30],
            'pca__n_components': [40, 80, 120],
            'clf__var_smoothing': [1e-9, 1e-8, 1e-7],
        }),
    },
    {
        'name': 'E07_Preproc_FastText_GaussianNB',
        'search_kind': 'random',
        'n_iter': rand_iter,
        'pipeline': Pipeline([
            ('prep', TextPreprocessor()),
            ('emb', MeanGensimEmbeddingTransformer(method='fasttext', seed=SEED, workers=1)),
            ('clf', GaussianNB()),
        ]),
        'params': add_preproc_variants({
            'emb__vector_size': [100, 150, 200],
            'emb__window': [5, 8],
            'emb__min_count': [1, 2],
            'emb__epochs': [20, 30],
            'clf__var_smoothing': [1e-9, 1e-8, 1e-7],
        }),
    },
    {
        'name': 'E08_Preproc_FastText_PCA_GaussianNB',
        'search_kind': 'random',
        'n_iter': rand_iter,
        'pipeline': Pipeline([
            ('prep', TextPreprocessor()),
            ('emb', MeanGensimEmbeddingTransformer(method='fasttext', vector_size=200, seed=SEED, workers=1)),
            ('pca', PCA(random_state=SEED)),
            ('clf', GaussianNB()),
        ]),
        'params': add_preproc_variants({
            'emb__window': [5, 8],
            'emb__min_count': [1, 2],
            'emb__epochs': [20, 30],
            'pca__n_components': [40, 80, 120],
            'clf__var_smoothing': [1e-9, 1e-8, 1e-7],
        }),
    },

    # --- Embeddings clasicos + CatBoost/LightGBM ---
    {
        'name': 'E09_Preproc_Word2Vec_CatBoost',
        'search_kind': 'random',
        'n_iter': rand_iter,
        'pipeline': Pipeline([
            ('prep', TextPreprocessor()),
            ('emb', MeanGensimEmbeddingTransformer(method='word2vec', seed=SEED, workers=1)),
            ('clf', CatBoostClassifier(
                loss_function='MultiClass',
                random_state=SEED,
                verbose=0,
                allow_writing_files=False,
                thread_count=-1,
            )),
        ]),
        'params': add_preproc_variants({
            'emb__vector_size': [100, 150, 200],
            'emb__window': [5, 8],
            'emb__min_count': [1, 2],
            'emb__epochs': [20, 30],
            'clf__depth': [6, 8],
            'clf__learning_rate': [0.03, 0.05, 0.1],
            'clf__iterations': cb_iters,
        }),
    },
    {
        'name': 'E10_Preproc_Word2Vec_PCA_LightGBM',
        'search_kind': 'random',
        'n_iter': rand_iter,
        'pipeline': Pipeline([
            ('prep', TextPreprocessor()),
            ('emb', MeanGensimEmbeddingTransformer(method='word2vec', vector_size=200, seed=SEED, workers=1)),
            ('pca', PCA(random_state=SEED)),
            ('clf', LGBMClassifier(
                objective='multiclass',
                num_class=num_classes,
                random_state=SEED,
                n_jobs=-1,
                verbosity=-1,
            )),
        ]),
        'params': add_preproc_variants({
            'emb__window': [5, 8],
            'emb__min_count': [1, 2],
            'emb__epochs': [20, 30],
            'pca__n_components': [40, 80, 120],
            'clf__num_leaves': [31, 63],
            'clf__learning_rate': [0.03, 0.05, 0.1],
            'clf__n_estimators': lgb_estimators,
        }),
    },
    {
        'name': 'E11_Preproc_FastText_LightGBM',
        'search_kind': 'random',
        'n_iter': rand_iter,
        'pipeline': Pipeline([
            ('prep', TextPreprocessor()),
            ('emb', MeanGensimEmbeddingTransformer(method='fasttext', seed=SEED, workers=1)),
            ('clf', LGBMClassifier(
                objective='multiclass',
                num_class=num_classes,
                random_state=SEED,
                n_jobs=-1,
                verbosity=-1,
            )),
        ]),
        'params': add_preproc_variants({
            'emb__vector_size': [100, 150, 200],
            'emb__window': [5, 8],
            'emb__min_count': [1, 2],
            'emb__epochs': [20, 30],
            'clf__num_leaves': [31, 63],
            'clf__learning_rate': [0.03, 0.05, 0.1],
            'clf__n_estimators': lgb_estimators,
        }),
    },
    {
        'name': 'E12_Preproc_FastText_PCA_CatBoost',
        'search_kind': 'random',
        'n_iter': rand_iter,
        'pipeline': Pipeline([
            ('prep', TextPreprocessor()),
            ('emb', MeanGensimEmbeddingTransformer(method='fasttext', vector_size=200, seed=SEED, workers=1)),
            ('pca', PCA(random_state=SEED)),
            ('clf', CatBoostClassifier(
                loss_function='MultiClass',
                random_state=SEED,
                verbose=0,
                allow_writing_files=False,
                thread_count=-1,
            )),
        ]),
        'params': add_preproc_variants({
            'emb__window': [5, 8],
            'emb__min_count': [1, 2],
            'emb__epochs': [20, 30],
            'pca__n_components': [40, 80, 120],
            'clf__depth': [6, 8],
            'clf__learning_rate': [0.03, 0.05, 0.1],
            'clf__iterations': cb_iters,
        }),
    },
]

if ENABLE_HF_EXPERIMENTS:
    hf_experiments = [
        # Solo 2 experimentos avanzados: sin PCA y con PCA
        {
            'name': 'E13_Preproc_HF_GaussianNB',
            'search_kind': 'random',
            'n_iter': HF_N_ITER,
            'pipeline': Pipeline([
                ('prep', TextPreprocessor()),
                ('emb', HFEmbeddingTransformer(
                    model_name=HF_MODEL_NAME,
                    device=HF_DEVICE,
                    batch_size=HF_BATCH_SIZE,
                    use_cache=True,
                    cache_dir='cache/hf_embeddings',
                )),
                ('clf', GaussianNB()),
            ]),
            'params': add_preproc_variants({
                'emb__normalize_embeddings': [True, False],
                'clf__var_smoothing': [1e-9, 1e-8, 1e-7],
            }),
        },
        {
            'name': 'E14_Preproc_HF_PCA_GaussianNB',
            'search_kind': 'random',
            'n_iter': HF_N_ITER,
            'pipeline': Pipeline([
                ('prep', TextPreprocessor()),
                ('emb', HFEmbeddingTransformer(
                    model_name=HF_MODEL_NAME,
                    device=HF_DEVICE,
                    batch_size=HF_BATCH_SIZE,
                    use_cache=True,
                    cache_dir='cache/hf_embeddings',
                )),
                ('pca', PCA(random_state=SEED)),
                ('clf', GaussianNB()),
            ]),
            'params': add_preproc_variants({
                'emb__normalize_embeddings': [True, False],
                'pca__n_components': [64, 128],
                'clf__var_smoothing': [1e-9, 1e-8, 1e-7],
            }),
        },
    ]
    experiments.extend(hf_experiments)

if MAX_EXPERIMENTS > 0:
    experiments = experiments[:MAX_EXPERIMENTS]

print('Numero de experimentos:', len(experiments))
for e in experiments:
    print('-', e['name'])

Numero de experimentos: 13
- E01_Preproc_TFIDF_ComplementNB
- E01b_Preproc_TFIDF_SVD_ComplementNB
- E02_Preproc_TFIDF_SVD_CatBoost
- E03_Preproc_TFIDF_SVD_LightGBM
- E04_Preproc_TFIDF_NMF_LightGBM
- E05_Preproc_Word2Vec_GaussianNB
- E06_Preproc_Word2Vec_PCA_GaussianNB
- E07_Preproc_FastText_GaussianNB
- E08_Preproc_FastText_PCA_GaussianNB
- E09_Preproc_Word2Vec_CatBoost
- E10_Preproc_Word2Vec_PCA_LightGBM
- E11_Preproc_FastText_LightGBM
- E12_Preproc_FastText_PCA_CatBoost


In [12]:
# Ejecucion de experimentos
results = []
search_map = {}

for exp in experiments:
    try:
        row, search = run_experiment(exp)
        results.append(row)
        search_map[exp['name']] = search
    except Exception as exc:
        print(f"Error en {exp['name']}: {exc}")
        results.append({
            'experiment': exp['name'],
            'status': 'error',
            'error': str(exc),
        })

results_df = pd.DataFrame(results)
results_df


===== E01_Preproc_TFIDF_ComplementNB =====
VAL f1_macro : 0.8238
TEST f1_macro: 0.8406
Candidatos   : 3
Tiempo (s)   : 6.09

===== E01b_Preproc_TFIDF_SVD_ComplementNB =====
Error en E01b_Preproc_TFIDF_SVD_ComplementNB: Negative values in data passed to ComplementNB (input X).

===== E02_Preproc_TFIDF_SVD_CatBoost =====


KeyboardInterrupt: 

In [ ]:
# Ranking comparativo y seleccion del ganador
ok_df = results_df[results_df['status'] == 'ok'].copy()

if ok_df.empty:
    raise RuntimeError('Ningun experimento finalizo correctamente.')

# Ganador definido SOLO por test_f1_macro
ok_df = ok_df.sort_values(
    by=['test_f1_macro'],
    ascending=False,
).reset_index(drop=True)

display_cols = [
    'experiment',
    'val_f1_macro',
    'val_f1_weighted',
    'val_accuracy',
    'test_f1_macro',
    'test_f1_weighted',
    'test_accuracy',
    'fit_time_sec',
    'n_candidates',
]

print('Ranking de experimentos (criterio: test_f1_macro):')
display(ok_df[display_cols])

winner_name = ok_df.iloc[0]['experiment']
print('Experimento ganador:', winner_name)

Ranking de experimentos (criterio: test_f1_macro):


,experiment,val_f1_macro,val_f1_weighted,val_accuracy,test_f1_macro,test_f1_weighted,test_accuracy,fit_time_sec,n_candidates
0,E01_Preproc_TFIDF_ComplementNB,0.841114,0.866912,0.873016,0.861123,0.878051,0.882678,73.88,8
1,E14_Preproc_HF_PCA_GaussianNB,0.817589,0.844908,0.843340,0.834470,0.860749,0.859903,57.96,4
2,E13_Preproc_HF_GaussianNB,0.807231,0.833935,0.832298,0.831967,0.857295,0.855072,92.92,4
3,E04_Preproc_TFIDF_NMF_LightGBM,0.827061,0.854437,0.856453,0.817785,0.845071,0.847481,1288.14,8
4,E03_Preproc_TFIDF_SVD_LightGBM,0.811072,0.846357,0.848861,0.809609,0.845513,0.846101,308.89,8
5,E02_Preproc_TFIDF_SVD_CatBoost,0.806162,0.833919,0.835059,0.800249,0.834018,0.835749,299.62,8
6,E09_Preproc_Word2Vec_CatBoost,0.785216,0.819516,0.820566,0.779137,0.823042,0.824707,264.66,8
7,E10_Preproc_Word2Vec_PCA_LightGBM,0.774391,0.810299,0.812974,0.775057,0.817968,0.820566,347.79,8
8,E06_Preproc_Word2Vec_PCA_GaussianNB,0.769413,0.807643,0.803313,0.767919,0.810436,0.806073,151.62,8
9,E08_Preproc_FastText_PCA_GaussianNB,0.753796,0.795110,0.790890,0.760158,0.800172,0.797792,801.59,8


Experimento ganador: E01_Preproc_TFIDF_ComplementNB


## Modelo ganador

In [ ]:
# Evaluacion detallada del ganador
winner_result = search_map[winner_name]
winner_model = winner_result['best_model']

y_val_pred = winner_model.predict(X_val)
y_test_pred = winner_model.predict(X_test)

print('Best params del ganador:')
print(winner_result['best_params'])
print('Numero de candidatos evaluados:', winner_result['n_candidates'])

print('\nClassification report - VAL')
print(classification_report(y_val, y_val_pred, digits=4))

print('\nClassification report - TEST')
print(classification_report(y_test, y_test_pred, digits=4))

cm = confusion_matrix(y_test, y_test_pred)
fig, ax = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay(cm).plot(ax=ax, colorbar=False)
ax.set_title(f'Matriz de confusion TEST - {winner_name}')
plt.tight_layout()
plt.show()

Best params del ganador:
{'tfidf__sublinear_tf': True, 'tfidf__ngram_range': (1, 2), 'tfidf__min_df': 2, 'prep__stem': False, 'prep__remove_stopwords': True, 'prep__lemmatize': True, 'clf__alpha': 0.3}
Numero de candidatos evaluados: 8

Classification report - VAL
              precision    recall  f1-score   support

           1     0.7632    0.7632    0.7632        76
           2     0.7895    0.8182    0.8036        55
           3     0.9343    0.9552    0.9446       134
           4     0.8580    0.9805    0.9152       154
           5     0.8848    0.9125    0.8985       160
           6     0.9252    0.9519    0.9384       104
           7     0.8286    0.9831    0.8992       118
           8     0.7778    0.4179    0.5437        67
           9     0.8500    0.6538    0.7391        52
          10     0.7750    0.5849    0.6667        53
          11     0.9167    0.8462    0.8800        91
          12     1.0000    0.7447    0.8537        47
          13     0.8400    0.900

### Interpretacion de la matriz de confusion

La matriz de confusion permite identificar patrones de error sistematicos. Se espera observar confusiones entre ODS tematicamente relacionados — por ejemplo, entre ODS 1 (Fin de la pobreza) y ODS 2 (Hambre cero), o entre ODS 14 (Vida submarina) y ODS 15 (Vida de ecosistemas terrestres) — porque sus textos comparten vocabulario semanticamente cercano. Las confusiones entre ODS distantes tematicamente indicarian problemas de representacion o insuficiencia de datos.

En el `classification_report`, clases con bajo `recall` señalan ODS cuyo vocabulario distintivo es insuficiente o se solapa con otros, mientras que bajo `precision` indica que el modelo atrae textos de ODS vecinos. Ambas metricas, junto con el `f1_macro`, permiten evaluar si el modelo es equitativo entre todas las clases.

### Analisis de reduccion de dimensionalidad: varianza explicada y visualizacion t-SNE

Para validar que la reduccion de dimensionalidad preserva la estructura discriminativa del espacio de features, se realizan dos analisis sobre la representacion TF-IDF del ganador:

1. **Varianza explicada acumulada (TruncatedSVD)**: se grafica la proporcion de varianza explicada por componente. Permite verificar cuantos componentes son necesarios para retener, por ejemplo, el 90-95% de la varianza total, sirviendo como criterio para elegir `n_components`. Es la misma metodologia empleada para seleccionar componentes PCA en datasets de imagenes, adaptada aqui a la representacion textual.

2. **Visualizacion t-SNE 2D**: se aplica TruncatedSVD como primer paso de reduccion de ruido (similar a aplicar PCA antes de t-SNE como se estudio para datos densos) y luego t-SNE para proyectar a 2 dimensiones. Esto permite visualizar si las clases ODS forman agrupaciones separadas en el espacio latente. Una buena separacion visual confirma que la representacion captura diferencias semanticas entre ODS.

In [ ]:
# Varianza explicada acumulada (TruncatedSVD sobre TF-IDF del ganador)
from sklearn.manifold import TSNE

# Reconstruir la representacion TF-IDF con el preprocesamiento del ganador
prep_step = winner_model.named_steps.get('prep')
tfidf_step = winner_model.named_steps.get('tfidf')

if prep_step is not None and tfidf_step is not None:
    X_train_prep = prep_step.transform(X_train)
    X_test_prep = prep_step.transform(X_test)
    X_train_tfidf = tfidf_step.transform(X_train_prep)
    X_test_tfidf = tfidf_step.transform(X_test_prep)

    # SVD con suficientes componentes para analizar varianza
    n_max = min(300, X_train_tfidf.shape[1] - 1)
    svd_analysis = TruncatedSVD(n_components=n_max, random_state=SEED)
    svd_analysis.fit(X_train_tfidf)

    cumvar = np.cumsum(svd_analysis.explained_variance_ratio_)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Varianza explicada acumulada
    axes[0].plot(range(1, n_max + 1), cumvar, linewidth=1.5)
    axes[0].axhline(y=0.90, color='r', linestyle='--', alpha=0.7, label='90% varianza')
    axes[0].axhline(y=0.95, color='orange', linestyle='--', alpha=0.7, label='95% varianza')
    n90 = int(np.searchsorted(cumvar, 0.90) + 1)
    n95 = int(np.searchsorted(cumvar, 0.95) + 1)
    axes[0].axvline(x=n90, color='r', linestyle=':', alpha=0.5)
    axes[0].axvline(x=n95, color='orange', linestyle=':', alpha=0.5)
    axes[0].set_xlabel('Numero de componentes SVD')
    axes[0].set_ylabel('Varianza explicada acumulada')
    axes[0].set_title('Varianza explicada acumulada (TruncatedSVD sobre TF-IDF)')
    axes[0].legend()
    axes[0].text(n90 + 3, 0.88, f'n={n90}', fontsize=9, color='r')
    axes[0].text(n95 + 3, 0.93, f'n={n95}', fontsize=9, color='orange')

    # t-SNE 2D sobre SVD reducido (100 componentes como paso de reduccion de ruido)
    svd_reduced = TruncatedSVD(n_components=100, random_state=SEED)
    X_test_svd = svd_reduced.fit_transform(X_test_tfidf)

    tsne = TSNE(n_components=2, random_state=SEED, perplexity=30, max_iter=1000)
    X_test_2d = tsne.fit_transform(X_test_svd)

    scatter = axes[1].scatter(
        X_test_2d[:, 0], X_test_2d[:, 1],
        c=y_test.values, cmap='tab20', alpha=0.6, s=12,
    )
    axes[1].set_xlabel('t-SNE dim 1')
    axes[1].set_ylabel('t-SNE dim 2')
    axes[1].set_title('Visualizacion t-SNE 2D del espacio TF-IDF (test)')
    cbar = plt.colorbar(scatter, ax=axes[1], ticks=sorted(y_test.unique()))
    cbar.set_label('ODS')

    plt.tight_layout()
    plt.show()

    print(f'Componentes para 90% varianza: {n90}')
    print(f'Componentes para 95% varianza: {n95}')
else:
    print('El modelo ganador no tiene pasos prep/tfidf; omitiendo analisis SVD/t-SNE.')

In [ ]:
# Evidencia de 4 predicciones de test
rng = np.random.default_rng(SEED)
idx = rng.choice(np.arange(len(X_test)), size=4, replace=False)

X_test_reset = X_test.reset_index(drop=True)
y_test_reset = y_test.reset_index(drop=True)
y_pred_reset = pd.Series(y_test_pred)

proba = None
if hasattr(winner_model, 'predict_proba'):
    try:
        proba = winner_model.predict_proba(X_test)
    except Exception:
        proba = None

rows = []
for i in idx:
    row = {
        'texto_preview': X_test_reset.iloc[i][:260].replace('\n', ' ') + ('...' if len(X_test_reset.iloc[i]) > 260 else ''),
        'ODS_real': int(y_test_reset.iloc[i]),
        'ODS_predicho': int(y_pred_reset.iloc[i]),
    }
    if proba is not None:
        row['confianza_max'] = float(np.max(proba[i]))
    rows.append(row)

pd.DataFrame(rows)

,texto_preview,ODS_real,ODS_predicho,confianza_max
0,La vivienda también puede resultar inasequible...,11,1,0.099168
1,"Sin embargo, aunque indica que el agua no esca...",6,6,0.308725
2,El aumento de la movilidad internacional de lo...,3,3,0.173811
3,En los cinco países se han tomado medidas para...,5,5,0.156929


In [ ]:
# Exportables
artifacts_dir = Path('artifacts')
artifacts_dir.mkdir(parents=True, exist_ok=True)

ok_df.to_csv(artifacts_dir / 'experiment_results.csv', index=False)

try:
    joblib.dump(winner_model, artifacts_dir / 'best_model.joblib')
    model_saved = True
except Exception as exc:
    model_saved = False
    print('No se pudo serializar el modelo ganador con joblib:', exc)

metadata = {
    'winner_experiment': winner_name,
    'best_params': winner_result['best_params'],
    'n_candidates': winner_result['n_candidates'],
    'val_f1_macro': float(f1_score(y_val, y_val_pred, average='macro')),
    'test_f1_macro': float(f1_score(y_test, y_test_pred, average='macro')),
    'seed': SEED,
    'train_size': int(len(X_train)),
    'val_size': int(len(X_val)),
    'test_size': int(len(X_test)),
    'hf_experiments_enabled': ENABLE_HF_EXPERIMENTS,
    'hf_device': HF_DEVICE,
    'model_saved_joblib': model_saved,
}

(artifacts_dir / 'best_model_metadata.json').write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

print('Exportables generados en:', artifacts_dir.resolve())
print('best_model.joblib guardado:', model_saved)

Exportables generados en: /Users/adrianalarcon/Library/CloudStorage/OneDrive-UniversidaddelosAndes/MAIA/aprendizaje_no_supervisado/assignment_2/artifacts
best_model.joblib guardado: True


## Notas operativas

- Para corrida completa: dejar variables por defecto.
- Para smoke test rapido:
  - `USE_DEV_SAMPLE=1`
  - `FAST_MODE=1`
  - `MAX_EXPERIMENTS=3`
  - `ENABLE_HF_EXPERIMENTS=0` (si no hay MPS)
  - `LIMIT_PREPROC_VARIANTS=1`

Optimizacion de embeddings avanzados:
- solo 2 experimentos HF (`E13` sin PCA y `E14` con PCA),
- un unico modelo HF (`HF_MODEL_NAME`),
- cache activo en `cache/hf_embeddings`,
- `HF_N_ITER` bajo para reducir tiempo.

Nota: este notebook no usa CV; el tuning se hace solo con `train` y seleccion por `val`.